# Robust relevance-pursuit GP

This notebook compares `RobustRelevancePursuitSingleTaskGP` with a standard `SingleTaskGP` on data containing a small number of strong outliers.

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import SingleTaskGP, RobustRelevancePursuitSingleTaskGP

torch.manual_seed(0)
dtype = torch.double

## 1. Synthetic data with outliers

In [ ]:
train_X = torch.linspace(0, 1, 28, dtype=dtype).unsqueeze(-1)
true_Y = torch.sin(2 * torch.pi * train_X)
train_Y = true_Y + 0.04 * torch.randn_like(true_Y)
outlier_idx = torch.tensor([5, 17, 23])
train_Y[outlier_idx] += torch.tensor([[1.4], [-1.6], [1.2]], dtype=dtype)
train_X.shape, train_Y.shape

## 2. Standard GP baseline

In [ ]:
baseline = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(baseline.make_mll())

## 3. Robust relevance-pursuit GP

In [ ]:
robust = RobustRelevancePursuitSingleTaskGP(
    train_X,
    train_Y,
    convex_parameterization=True,
)
print(robust.raw_train_X.shape, robust.raw_train_Y.shape)
print('supports_mll =', robust.supports_mll)
fit_gpytorch_mll(robust.make_mll())

## 4. Posterior comparison

In [ ]:
test_X = torch.linspace(0, 1, 200, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    p_base = baseline.posterior(test_X)
    p_robust = robust.posterior(test_X)
m_base = p_base.mean.squeeze(-1)
m_robust = p_robust.mean.squeeze(-1)
s_robust = p_robust.variance.sqrt().squeeze(-1)
plt.figure(figsize=(8, 4))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label='observations')
plt.scatter(train_X[outlier_idx].squeeze(-1), train_Y[outlier_idx].squeeze(-1), marker='x', s=80, label='outliers')
plt.plot(test_X.squeeze(-1), m_base, label='SingleTaskGP')
plt.plot(test_X.squeeze(-1), m_robust, label='Robust GP')
plt.fill_between(test_X.squeeze(-1), m_robust - 2*s_robust, m_robust + 2*s_robust, alpha=0.2)
plt.legend()
plt.xlabel('x')
plt.ylabel('y')
plt.title('Effect of outliers on GP regression')
plt.show()

## 5. When to use / not use

- Use this model when a small subset of observations may be corrupted or unusually influential.
- Compare against a standard GP instead of assuming robust modeling is always better.
- If apparent outliers are actually a separate regime, a mixture, task, or conditional model may be more appropriate.
- robotorchan keeps the standard exact-GP `make_mll()` interface for this wrapper.